# Pandas and Parquet

This notebook introduces Parquet through the pandas API. It first creates a small e-commerce dataset, writes it to Parquet, and reads it back. It then converts the MovieLens `movies.csv` and `ratings.csv` files into Parquet.

## Installation

Pandas needs a Parquet engine. This notebook uses PyArrow:

```powershell
python -m pip install pandas pyarrow
```

Output locations:

- E-commerce example: `C:\data\output\ecommerce_orders.parquet`
- MovieLens files: `C:\data\movielens\parquet`


## 1. Import pandas and prepare directories

`DataFrame.to_parquet()` writes a DataFrame, while `pd.read_parquet()` reads it. The `pyarrow` engine performs the Parquet serialization.


In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow as pa

OUTPUT_DIR = Path(r"C:\data\output")
MOVIELENS_ROOT = Path(r"C:\data\movielens")
MOVIELENS_SOURCE = MOVIELENS_ROOT / "ml-latest-small"
MOVIELENS_PARQUET_DIR = MOVIELENS_ROOT / "parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MOVIELENS_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

print(f"pandas version: {pd.__version__}")
print(f"PyArrow version: {pa.__version__}")
print(f"E-commerce output: {OUTPUT_DIR}")
print(f"MovieLens output: {MOVIELENS_PARQUET_DIR}")


## 2. Create an e-commerce DataFrame

Good data types matter. Integers, floating-point values, booleans, strings, and timestamps map naturally to typed Parquet columns.


In [ ]:
orders = pd.DataFrame({
    "order_id": pd.Series([1001, 1002, 1003, 1004, 1005, 1006], dtype="int64"),
    "customer_id": pd.Series([501, 502, 501, 503, 504, 502], dtype="int64"),
    "category": pd.Series(
        ["Electronics", "Books", "Home", "Electronics", "Books", "Home"],
        dtype="string",
    ),
    "quantity": pd.Series([1, 2, 1, 1, 3, 2], dtype="int32"),
    "unit_price": pd.Series([899.99, 15.50, 42.25, 129.00, 8.99, 25.00], dtype="float64"),
    "is_member": pd.Series([True, False, True, True, False, False], dtype="bool"),
    "ordered_at": pd.to_datetime([
        "2026-08-01T09:15:00Z",
        "2026-08-01T10:30:00Z",
        "2026-08-02T11:00:00Z",
        "2026-08-02T14:45:00Z",
        "2026-08-03T08:20:00Z",
        "2026-08-03T16:10:00Z",
    ], utc=True),
})

orders["order_total"] = orders["quantity"] * orders["unit_price"]

print(orders)
print("\nData types:")
print(orders.dtypes)


## 3. Write the e-commerce data to Parquet

- `engine="pyarrow"` selects the Parquet implementation explicitly.
- `compression="snappy"` provides fast, widely supported compression.
- `index=False` prevents pandas from storing the DataFrame's default row index as a data column.


In [ ]:
orders_parquet_path = OUTPUT_DIR / "ecommerce_orders.parquet"

orders.to_parquet(
    orders_parquet_path,
    engine="pyarrow",
    compression="snappy",
    index=False,
)

print(f"Created: {orders_parquet_path}")
print(f"File size: {orders_parquet_path.stat().st_size:,} bytes")


## 4. Read the e-commerce Parquet file

The reconstructed DataFrame retains column names and types. We validate the full round trip and also demonstrate reading only selected columns.


In [ ]:
orders_from_parquet = pd.read_parquet(orders_parquet_path, engine="pyarrow")

print(orders_from_parquet)
print("\nData types after reading:")
print(orders_from_parquet.dtypes)

pd.testing.assert_frame_equal(orders_from_parquet, orders)
print("\nE-commerce round-trip validation passed.")

order_summary = pd.read_parquet(
    orders_parquet_path,
    engine="pyarrow",
    columns=["order_id", "category", "order_total"],
)
print("\nSelected columns only:")
print(order_summary)


# MovieLens CSV-to-Parquet conversion

The source files are:

- `C:\data\movielens\ml-latest-small\movies\movies.csv`
- `C:\data\movielens\ml-latest-small\ratings\ratings.csv`

The converted files will be placed in `C:\data\movielens\parquet`.


## 5. Locate and validate the source CSV files

Failing early with a clear message is better than allowing a later `read_csv` call to fail ambiguously.


In [ ]:
movies_csv_path = MOVIELENS_SOURCE / "movies" / "movies.csv"
ratings_csv_path = MOVIELENS_SOURCE / "ratings" / "ratings.csv"

for source_path in (movies_csv_path, ratings_csv_path):
    if not source_path.is_file():
        raise FileNotFoundError(f"Required MovieLens source file not found: {source_path}")
    print(f"Found {source_path} ({source_path.stat().st_size:,} bytes)")


## 6. Read the MovieLens CSV files with explicit types

Explicit types make the intended schema clear and avoid accidental type inference. Movie IDs and user IDs fit comfortably in 32-bit integers in this dataset. Ratings use 32-bit floating-point values, and the source timestamp remains a 64-bit Unix timestamp.


In [ ]:
movies = pd.read_csv(
    movies_csv_path,
    dtype={"movieId": "int32", "title": "string", "genres": "string"},
)

ratings = pd.read_csv(
    ratings_csv_path,
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "rating": "float32",
        "timestamp": "int64",
    },
)

print(f"Movies: {movies.shape[0]:,} rows × {movies.shape[1]} columns")
print(movies.head())
print("\nMovie types:")
print(movies.dtypes)

print(f"\nRatings: {ratings.shape[0]:,} rows × {ratings.shape[1]} columns")
print(ratings.head())
print("\nRating types:")
print(ratings.dtypes)


## 7. Write MovieLens DataFrames to Parquet

Each CSV becomes one Parquet file. Snappy is a sensible introductory default because it prioritizes fast compression and decompression.


In [ ]:
movies_parquet_path = MOVIELENS_PARQUET_DIR / "movies.parquet"
ratings_parquet_path = MOVIELENS_PARQUET_DIR / "ratings.parquet"

movies.to_parquet(
    movies_parquet_path,
    engine="pyarrow",
    compression="snappy",
    index=False,
)
ratings.to_parquet(
    ratings_parquet_path,
    engine="pyarrow",
    compression="snappy",
    index=False,
)

print(f"Created {movies_parquet_path} ({movies_parquet_path.stat().st_size:,} bytes)")
print(f"Created {ratings_parquet_path} ({ratings_parquet_path.stat().st_size:,} bytes)")


## 8. Read and validate the converted Parquet files

The assertions verify that the written Parquet data matches the source DataFrames, including column order and pandas data types.


In [ ]:
movies_check = pd.read_parquet(movies_parquet_path, engine="pyarrow")
ratings_check = pd.read_parquet(ratings_parquet_path, engine="pyarrow")

pd.testing.assert_frame_equal(movies_check, movies)
pd.testing.assert_frame_equal(ratings_check, ratings)

print("MovieLens round-trip validation passed.")
print(f"Movies read from Parquet: {len(movies_check):,}")
print(f"Ratings read from Parquet: {len(ratings_check):,}")

print("\nAverage rating by movie (first five results):")
average_ratings = (
    ratings_check.groupby("movieId", as_index=False)["rating"]
    .mean()
    .rename(columns={"rating": "average_rating"})
    .head()
)
print(average_ratings)


## 9. Compare CSV and Parquet sizes

File size depends on the data, selected compression codec, and writer settings. Parquet often compresses analytical data well because values from the same column are stored together.


In [ ]:
size_comparison = pd.DataFrame({
    "dataset": ["movies", "ratings"],
    "csv_bytes": [movies_csv_path.stat().st_size, ratings_csv_path.stat().st_size],
    "parquet_bytes": [movies_parquet_path.stat().st_size, ratings_parquet_path.stat().st_size],
})
size_comparison["parquet_vs_csv_pct"] = (
    size_comparison["parquet_bytes"] / size_comparison["csv_bytes"] * 100
).round(1)

print(size_comparison)


## Summary

- `DataFrame.to_parquet()` writes pandas data to Parquet.
- `pd.read_parquet()` reconstructs a DataFrame and supports column projection.
- Use `index=False` when the DataFrame index is not meaningful data.
- Explicit CSV types produce a predictable Parquet schema.
- The MovieLens CSV files were converted and validated through a full round trip.

Generated files:

- `C:\data\output\ecommerce_orders.parquet`
- `C:\data\movielens\parquet\movies.parquet`
- `C:\data\movielens\parquet\ratings.parquet`
